### Análisis Exploratorio de Datos (EDA)

#### Carga de librerías e inicialización

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Carga de dataset limpio
df_spotify_clean = pd.read_csv("../data/processed/spotify_2000_clean.csv")

# Verificación estructura de datos
df_spotify_clean.info()

In [ ]:
# Verificación tamaño dataset y valores faltantes
print("Tamaño de dataset:", df_spotify_clean.shape)
print("\n Número de valores ausentes:")
print(df_spotify_clean.isnull().sum())

# Resumen estadístico de las variables acústicas y popularidad
acoustic_features = [
    'beats_per_minute_bpm', 'energy', 'danceability', 'loudness_db', 
    'liveness', 'valence', 'length_duration', 'acousticness', 'speechiness', 'popularity'
]

print("\nResumen estadístico de las variables acústicas y popularidad:")
df_spotify_clean[acoustic_features].describe().round(1)

#### Distribución de popularidad

In [ ]:
#Histograma de popularidad general del catálogo
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
sns.histplot(data=df_spotify_clean['popularity'], kde=True, color='#1DB954', bins=30)
plt.title('Distribución de Popularity en el catálogo', fontsize=12, fontweight='bold') 
plt.xlabel('Puntuación de Popularidad')
plt.ylabel('Volumen de canciones')

# Boxplot popularidad
plt.subplot(1, 2, 2)
sns.boxplot(data=df_spotify_clean['popularity'], color='gray', 
            flierprops={"markerfacecolor":"blue", "marker":"o"})
plt.title('Boxplot de Diagnóstico: Popularity', fontsize=12, fontweight='bold')
plt.xlabel('Puntuación de Popularidad')

plt.tight_layout()
plt.show()

Hallazgo:  

Los valores de popularidad en todo el catálogo tienen como media general 59.4 puntos y una mediana de 62, con algunos outliers de valores bajos.

#### Análisis de distribución de variables acústicas


In [ ]:
# Lista de parámetros a evaluar
acoustic_parameters = [
    'beats_per_minute_bpm', 'energy', 'danceability', 'loudness_db', 
    'liveness', 'valence', 'length_duration', 'acousticness', 'speechiness'
]

# Configuración de gráficos de distribución
plt.figure(figsize=(18, 14))

for i, col in enumerate(acoustic_parameters):
    plt.subplot(3, 3, i + 1)
    sns.histplot(data=df_spotify_clean, x=col, kde=True, color='#1DB954', bins=30)
    plt.title(f'Distribución de {col}', fontsize=12, fontweight='bold')
    plt.xlabel('')
    plt.ylabel('')

plt.tight_layout()
plt.show()

# Boxplots para visualizar outliers en variables de escala abierta
plt.figure(figsize=(18, 5))
sub_outliers = ['length_duration', 'beats_per_minute_bpm', 'loudness_db']

for i, col in enumerate(sub_outliers):
    plt.subplot(1, 3, i + 1)
    sns.boxplot(data=df_spotify_clean, x=col, color='gray', 
                flierprops={"markerfacecolor":"blue", "marker":"o"})
    plt.title(f'Boxplot de Diagnóstico: {col}', fontsize=12, fontweight='bold')
    plt.xlabel('')

plt.tight_layout()
plt.show()
plt.tight_layout()
plt.show()

Hallazgo:

* Diagramas de dispersión

Las variables beats_per_minute_bpm y danceability tienen distribuciones normales concentradas en 120 BPM y 53.2.

energy y valence presentan distribuciones extendidas por todo el catálogo, esta última con sesgo hacia valores bajos.

lenght_duration se concentra masivamente entre los 200 y 300 segundos, con cola larga que se extiende hasta los 1400 segundos.

loudness_db esta sesgada a la derecha, la concentración del volumen está entre -10 db y -5 db con cola hacia la izquierda.

acousticness, liveness y speechiness presentan sesgo de valores muy bajos.



* Boxplots

Si bien la mediana de la duración de las canciones es de 245 segundos (4.1 minutos), los diagramas de caja muestran que length_duration posee valores extremadamente atípicos hacia la derecha los cuales superan los 1,000 segundos (16.7 minutos). 

A pesar de su distribución normal, beats_per_minute_bpm presenta outliers por debajo de 50 bpm y por encima de 190 BPM

loudness_db presenta outliers por debajo de los -18 db y llegan hasta los -27 db.


#### Correlación numérica de los atributos acústicos vs. Popularidad 

In [ ]:
# Definición de variables
columns_corr = acoustic_parameters + ['popularity']

# Matriz de correlación
matriz_corr = df_spotify_clean[columns_corr].corr()

# Mapa de calor
plt.figure(figsize=(10, 4))

sns.heatmap(matriz_corr, cmap='BrBG', vmax=1, vmin=-1, center=0, annot=True, fmt=".2f", linewidths=.5, 
    cbar_kws={"shrink": .8}
)

plt.title('Matriz de Correlación de Atributos acústicos vs. Popularidad', fontsize=14, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right')
plt.show()

La relación entre energy y loudness_db muestra un fuerte coeficiente positivo, en cambio con acousticness la correlación es negativa. Todos los atributos de manera individual muestran correlaciones entre 0.0 y 0.17 frente a la popularidad. 

#### Identificación de segmentos de alto y bajo rendimiento

In [ ]:
# Definición de los segmentos basados en volumen de género
# Umbral de alto rendimiento
percentil_75 = df_spotify_clean['popularity'].quantile(0.75) 
# Umbral de bajo rendimiento
percentil_25 = df_spotify_clean['popularity'].quantile(0.25) 

print(f"Corte alto rendimiento (Q3): >= {percentil_75}")
print(f"Corte bajo rendimiento (Q1): <= {percentil_25}\n")

# Segmento de alto rendimiento 
df_alto_rendimiento = df_spotify_clean[df_spotify_clean['popularity'] >= percentil_75]
top_generos_premium = df_alto_rendimiento['top_genre'].value_counts().head(5)

print("TOP 5 Géneros con mayor volumen en el segmento de alto rendimiento:")
print(top_generos_premium)
print("-" * 50)

# Segmento de bajo rendimiento
df_bajo_rendimiento = df_spotify_clean[df_spotify_clean['popularity'] <= percentil_25]
peores_generos = df_bajo_rendimiento['top_genre'].value_counts().head(5)

print("TOP 5 Géneros estancados en el segmento de bajo rendimiento:")
print(peores_generos)

El 25% de las canciones más exitosas tienen un puntaje de popularidad >= 71.0 y cualquier pista con una popularidad >= 41.29 se considera rezagada.

'album rock' se presenta como número uno en los dos segmentos. 'alternative rock', 'alternative metal' y 'adult standars' se consolidan en el top de alto rendimiento y 'ductch indie', 'dutch pop', 'dutch cabaret' y 'classic uk pop' en el top de bajo rendimiento.

#### Segmentación del catálogo por popularidad

In [ ]:
popularity = df_spotify_clean['popularity']

# Calcula cuartiles
q1 = popularity.quantile(0.25)
q2 = popularity.quantile(0.50)
q3 = popularity.quantile(0.75)

print(
    f"Q1 (25%): {q1:.1f} | "
    f"Q2 (50%): {q2:.1f} | "
    f"Q3 (75%): {q3:.1f}"
)

# Segmentación
df_spotify_clean['segment'] = pd.cut(
    df_spotify_clean['popularity'],
    bins=[
        popularity.min()-1,
        q1,
        q2,
        q3,
        popularity.max()
    ],
    labels=[
        '🔴 Bajo',
        '🟡 Medio-Bajo',
        '🟠 Medio-Alto',
        '🟢 Alto'
    ],
    include_lowest=True
)

# Resumen

segment_summary = (
    df_spotify_clean['segment']
    .value_counts()
    .sort_index()
    .reset_index()
)

segment_summary.columns = [
    'Segmento',
    'Canciones'
]

segment_summary['% del catálogo'] = (
    segment_summary['Canciones']
    / len(df_spotify_clean)
    * 100
).round(1)

# Resultado
print("\n SEGMENTACIÓN DEL CATÁLOGO")

print(
    segment_summary.to_string(
        index=False
    )
)

El catálogo de la plataforma se encuentra distribuido de forma equitativa en términos de volumen. El segmento de mayor rendimiento concentra solamente el 23% del catálogo.

#### 

#### Popularidad por género

In [ ]:
# Agrupación de datos 
genre_popularity = df_spotify_clean.groupby('top_genre').agg(
    total_songs=('popularity', 'count'),
    popularity_mean=('popularity', 'mean')
).sort_values(by='popularity_mean', ascending=False).head(20)

# Gráfico de barras 
plt.figure(figsize=(8, 4))
sns.barplot(data=genre_popularity.reset_index(), x='popularity_mean', y='top_genre', color='green')
plt.title('Top 20 Géneros con mayor popularidad promedio', fontsize=14, fontweight='bold')
plt.xlabel('Popularidad Promedio')
plt.ylabel('Género')
plt.show()

El top 20 de los géneros con mayor popularidad está liderado por 'celtic pukn' e 'indie pop', el resto del grupo presenta scores entre los 70 y 80 puntos.

#### Análsis de distribución de volumen por género

In [ ]:
# Número de géneros existentes
print('Número de géneros existentes:')
print(df_spotify_clean['top_genre'].nunique())


# Calcula el volumen de cada género
genre_count = df_spotify_clean['top_genre'].value_counts()

# Grafica el histograma de frecuencias de los tamaños
plt.figure(figsize=(8, 4))

sns.histplot(
    x=genre_count.values, 
    kde=True, 
    color='green' , 
    bins=25
)

plt.title('Distribución del volumen por género', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Número de canciones')
plt.ylabel('Número de Géneros de ese tamaño')

plt.tight_layout()
plt.show()

Esta gráfica de cola larga evidencia que una mínima cantidad de géneros acumulan un gran volúmen de canciones y más de 120 géneros, de un total de 149, no sobrepasan las 25 canciones.  

### Insights de negocio

* El grueso del catálogo comparte un patrón rítmico comercial estándar(beats_per_minute_bpm) y en la mayoría de las pistas predomina un perfil bailable (daceability). La plataforma también cuenta con una oferta equilibrada de intensidades (energy) y una ligera inclinación hacia estilos melancólicos (valence), como también el predominio de canciones grabadas en estudio (liveness), producciones electrónicas (acousticness) y contenido estrictamente musical (speechiness).

* Correlacionalidad: la relación casi nula entre los atributos y la popularidad no permite el uso de filtros simples o condicionales tradicionales para determinar el contenido de las listas premium. Entre las variables de mayor correlación para mitigar la redundancia se mantiene energy  como representante de la intensidad en lugar de loudness_db, por su mayor connotación musical y con escala cerrada.

* Género musical:  la asimetría en la distribución y el peso de las canciones del género invalida su uso como criterio de segmentación directa. Priorizar los géneros con mayor densidad de registros dejaría por fuera la gran diversidad musical del catálogo y si se incluyen canciones basándose solamente en el score de popularidad del género ignora la representatividad del su volumen.

* Para determinar qué canciones deberían hacer parte del catálogo premium es necesario apuntar hacia modelos avanzados que analicen el sello aústico global de cada pista y facilite procesar todas las variables en paralelo, permitiendo identificar patrones multidimensionales e interacciones no lineales que garanticen predicciones de éxito altamente precisas.

* Las variables clave elegidas para los modelos de Machine Learning son popularity como determinante del éxito comercial, energy (intensidad), danceability (adecuada para bailar), acousticness (acústica) como filtro de contraste para separar lo orgánico de lo digital y valence (positividad). Se descartan speechiness y liveness porque presentan sesgos extremos en su distribución y no aportan información útil para que el algoritmo diferencie una pista de otra. beats_per_minute_bpm y lenght_duration tampoco se tendrán en cuenta por sus escalas abiertas, presencia de outliers y nula contribución al aura musical.

* Filtros: se puede establecer un filtro de calidad inmediato a partir del corte de alto rendimiento Q3 (25% de las más escuchadas) para conformar la base de las listas premium y otro para optimizar la duración de las pistas con el fin de mitigar el churn.
